# Heart Disease Prediction with Machine Learning

## Objective

Build and evaluate a binary classification model that predicts whether a patient has heart disease from clinical attributes.

**Target**
- `0` — no heart disease
- `1` — heart disease

This notebook follows an end-to-end workflow: EDA → preprocessing → baseline models → hyperparameter tuning → evaluation → cross-validation → feature importance → error analysis.

**Dataset source:** [UCI Machine Learning Repository — Heart Disease](https://archive.ics.uci.edu/dataset/45/heart+disease)


## 1. Imports

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

## 2. Load data

In [ ]:
data_candidates = [
    Path("data/heart-disease.csv"),
    Path("../data/heart-disease.csv"),
]

DATA_PATH = next(path for path in data_candidates if path.exists())
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
df.head()

## 3. Exploratory Data Analysis

First, inspect the target distribution, data types, missing values, descriptive statistics, and relationships between selected variables.

In [ ]:
print("Target distribution:")
print(df["target"].value_counts())

print("\nMissing values:")
print(df.isna().sum())

df.describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df["target"].value_counts().sort_index().plot(
    kind="bar", ax=axes[0], title="Target Distribution"
)
axes[0].set_xlabel("Target")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=0)

pd.crosstab(df["cp"], df["target"]).plot(
    kind="bar", ax=axes[1], title="Heart Disease by Chest Pain Type"
)
axes[1].set_xlabel("Chest Pain Type")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
corr_matrix = df.corr(numeric_only=True)

plt.figure(figsize=(12, 8))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    linewidths=0.5,
    cmap="YlGnBu",
)
plt.title("Correlation Matrix")
plt.show()

corr_matrix["target"].drop("target").sort_values()

### EDA observations

The target is reasonably balanced. Several variables show noticeable association with the target, especially `cp`, `thalach`, `exang`, `oldpeak`, `ca`, `thal`, and `slope`. Correlation is used only as an exploratory signal and should not be interpreted as causation.

## 4. Train/test split

In [ ]:
X = df.drop(columns="target")
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

## 5. Baseline models

Logistic Regression and KNN are placed inside pipelines with `StandardScaler`. Random Forest does not require feature scaling.

In [ ]:
log_reg = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

knn = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier()),
])

models = {
    "Logistic Regression": log_reg,
    "KNN": knn,
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE),
}

baseline_scores = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    baseline_scores[name] = model.score(X_test, y_test)

pd.Series(baseline_scores, name="Test accuracy").sort_values(ascending=False)

## 6. Hyperparameter tuning

Hyperparameters are selected using cross-validation on the training set. The final test set is reserved for evaluation.

### 6.1 Logistic Regression

In [ ]:
log_reg_grid = {
    "model__C": np.logspace(-3, -0.5, 30),
    "model__solver": ["liblinear"],
}

gs_log_reg = GridSearchCV(
    estimator=log_reg,
    param_grid=log_reg_grid,
    cv=5,
    n_jobs=-1,
)

gs_log_reg.fit(X_train, y_train)

print("Best params:", gs_log_reg.best_params_)
print("Best CV score:", round(gs_log_reg.best_score_, 4))
print("Test score:", round(gs_log_reg.score(X_test, y_test), 4))

### 6.2 K-Nearest Neighbors

In [ ]:
knn_grid = {
    "model__n_neighbors": range(1, 21),
}

gs_knn = GridSearchCV(
    estimator=knn,
    param_grid=knn_grid,
    cv=5,
    n_jobs=-1,
)

gs_knn.fit(X_train, y_train)

print("Best params:", gs_knn.best_params_)
print("Best CV score:", round(gs_knn.best_score_, 4))
print("Test score:", round(gs_knn.score(X_test, y_test), 4))

### 6.3 Random Forest — coarse randomized search

In [ ]:
rf_grid = {
    "n_estimators": [100, 200, 300, 400, 500, 600, 750],
    "max_depth": [None, 3, 4, 5, 6, 8],
    "min_samples_split": [2, 8, 14, 16, 20],
    "min_samples_leaf": [1, 3, 5, 7, 9],
}

rs_rf = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE),
    param_distributions=rf_grid,
    n_iter=8,
    cv=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

rs_rf.fit(X_train, y_train)

print("Best randomized-search params:", rs_rf.best_params_)
print("Best CV score:", round(rs_rf.best_score_, 4))

### 6.4 Random Forest — refined grid search

In [ ]:
rf_grid_refined = {
    "n_estimators": [500, 750],
    "max_depth": [4, 5],
    "min_samples_split": [14, 16],
    "min_samples_leaf": [5, 7],
}

gs_rf = GridSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE),
    param_grid=rf_grid_refined,
    cv=5,
    n_jobs=-1,
)

gs_rf.fit(X_train, y_train)

print("Best params:", gs_rf.best_params_)
print("Best CV score:", round(gs_rf.best_score_, 4))
print("Test score:", round(gs_rf.score(X_test, y_test), 4))

## 7. Model comparison

In [ ]:
comparison = pd.DataFrame({
    "CV score": {
        "Logistic Regression": gs_log_reg.best_score_,
        "KNN": gs_knn.best_score_,
        "Random Forest": gs_rf.best_score_,
    },
    "Test accuracy": {
        "Logistic Regression": gs_log_reg.score(X_test, y_test),
        "KNN": gs_knn.score(X_test, y_test),
        "Random Forest": gs_rf.score(X_test, y_test),
    },
})

comparison

In [ ]:
comparison.plot(kind="bar", figsize=(9, 5))
plt.ylim(0, 1)
plt.ylabel("Score")
plt.title("Model Comparison")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

Random Forest achieved the strongest held-out test accuracy and was selected for detailed evaluation.

## 8. Final Random Forest evaluation

In [ ]:
best_rf = gs_rf.best_estimator_

y_pred = best_rf.predict(X_test)
y_prob = best_rf.predict_proba(X_test)[:, 1]

final_metrics = pd.Series({
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred),
    "ROC-AUC": roc_auc_score(y_test, y_prob),
})

final_metrics

In [ ]:
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title("Random Forest Confusion Matrix")
plt.show()

RocCurveDisplay.from_estimator(best_rf, X_test, y_test)
plt.title("Random Forest ROC Curve")
plt.show()

The model has high recall on the held-out test set, missing only one positive case, while producing several false positives. ROC-AUC measures ranking quality across thresholds and is not the same as accuracy.

## 9. Stratified cross-validation

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

scoring = ["accuracy", "precision", "recall", "f1"]

cv_results = cross_validate(
    best_rf,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
)

cv_metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1"],
    "Mean": [
        cv_results["test_accuracy"].mean(),
        cv_results["test_precision"].mean(),
        cv_results["test_recall"].mean(),
        cv_results["test_f1"].mean(),
    ],
    "Std": [
        cv_results["test_accuracy"].std(),
        cv_results["test_precision"].std(),
        cv_results["test_recall"].std(),
        cv_results["test_f1"].std(),
    ],
})

cv_metrics

Cross-validation is more informative than a single split for assessing stability. The relatively large standard deviations reflect the small dataset size.

## 10. Feature importance

In [ ]:
feature_importance = pd.Series(
    best_rf.feature_importances_,
    index=X.columns,
).sort_values(ascending=False)

feature_importance

In [ ]:
feature_importance.sort_values().plot.barh(figsize=(8, 6))
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Random Forest Feature Importance")
plt.tight_layout()
plt.show()

Feature importance describes how useful each variable was to the fitted Random Forest. It does **not** establish causality.

## 11. Error analysis

In [ ]:
results = X_test.copy()
results["y_true"] = y_test
results["y_pred"] = y_pred
results["probability_1"] = y_prob

results["error_type"] = np.select(
    [
        (results["y_true"] == 0) & (results["y_pred"] == 1),
        (results["y_true"] == 1) & (results["y_pred"] == 0),
    ],
    ["False Positive", "False Negative"],
    default="Correct",
)

results["error_type"].value_counts()

In [ ]:
errors = results[results["error_type"] != "Correct"].copy()
errors["confidence"] = np.maximum(
    errors["probability_1"],
    1 - errors["probability_1"],
)

errors[
    ["y_true", "y_pred", "probability_1", "confidence", "error_type"]
].sort_values("confidence", ascending=False)

Most test errors are false positives. The single false negative and the false positives are not extreme-confidence errors, which suggests that many mistakes occur in regions where class probabilities overlap.

## 12. Save the final model

In [ ]:
model_candidates = [
    Path("models/heart_disease_random_forest.joblib"),
    Path("../models/heart_disease_random_forest.joblib"),
]

MODEL_PATH = model_candidates[0] if Path("models").exists() else model_candidates[1]
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(best_rf, MODEL_PATH)
print("Saved:", MODEL_PATH)

## Conclusion

Three classification approaches were compared: Logistic Regression, KNN, and Random Forest. After hyperparameter tuning, Random Forest achieved the strongest held-out test accuracy.

### Final held-out test results

- **Accuracy:** ~0.82
- **Precision:** ~0.76
- **Recall:** ~0.97
- **F1-score:** ~0.85
- **ROC-AUC:** ~0.91

The most influential Random Forest features were `cp`, `thal`, `oldpeak`, `ca`, `thalach`, and `exang`.

Because the dataset is small, cross-validation results vary across folds. Future work could include a larger independent dataset, CatBoost/XGBoost, nested cross-validation, probability calibration, and permutation importance or SHAP.

> This project is for educational/portfolio purposes and is not intended for clinical decision-making.